In [25]:
# [셀 1] 설정 및 임포트
import pandas as pd
import numpy as np
import holidays   # pip install holidays

DATA_PATH = r'C:\Users\kjlee\도심관련\지하철관련\metro_usage_202607071800.csv'
OUT_DIR   = r'C:\Users\kjlee\도심관련\지하철관련'

In [26]:
# [셀 2] 원본 로드
# dtype=str: station_id, ticket_type은 숫자처럼 생긴 '코드'라 문자열로 읽어야 안전
df = pd.read_csv(
    DATA_PATH,
    dtype={'station_id': str, 'ticket_type': str},
    parse_dates=['bording_date'],
)
print(df.shape)          # (965734, 9) 이어야 정상
df.head(3)

(965734, 9)


,id,analy_type,dest,bording_date,ticket_type,bording_type,qty,station_id,bording_time
0,2350899,60min,NaN,2024-01-01,32,승차,5,1104,04:00:00
1,2350900,60min,NaN,2024-01-01,32,하차,5,1104,04:00:00
2,2350901,60min,NaN,2024-01-01,32,승차,2,1104,05:00:00


In [27]:
# [셀 3] 5min 해상도만 추출
# 원본에는 같은 데이터가 60/30/15/5min 4개 해상도로 중복 저장되어 있음(완전 중첩, 검증 완료)
# → 원천인 5min만 사용. dest(전체 결측), analy_type(필터 후 무의미), id(행번호)는 제거
raw = df[df.analy_type == '5min'].drop(columns=['dest', 'analy_type', 'id']).copy()

# 검증: 5min만으로도 전체 인원이 보존되는지 (전체 총합의 정확히 1/4이어야 함)
assert raw.qty.sum() * 4 == df.qty.sum(), '해상도 중첩 구조가 깨져 있음!'
print(f'{len(raw):,}행, 총 인원 {raw.qty.sum():,}명, 기간 {raw.bording_date.min().date()} ~ {raw.bording_date.max().date()}')

550,245행, 총 인원 16,267,347명, 기간 2024-01-01 ~ 2026-01-01


In [28]:
# [셀 4] 권종 합산 + 승차/하차 피벗
# 같은 5분 구간이 권종(보통권/카드코드)별로 여러 행 → 총 수요만 필요하므로 합산
agg = raw.groupby(['bording_date', 'bording_time', 'bording_type'], as_index=False).qty.sum()

# 날짜+시각 → 하나의 타임스탬프
agg['ts'] = pd.to_datetime(agg.bording_date.dt.strftime('%Y-%m-%d') + ' ' + agg.bording_time)

# 피벗: 세로로 쌓인 승차/하차 행 → 한 시점 한 행의 board/alight 두 컬럼 (값 변화 없음, 배치만)
wide = (agg.pivot(index='ts', columns='bording_type', values='qty')
           .rename(columns={'승차': 'board', '하차': 'alight'}))
print(wide.shape)
wide.head(3)

(166219, 2)


bording_type,board,alight
ts,,
2024-01-01 04:35:00,5.0,5.0
2024-01-01 05:10:00,1.0,NaN
2024-01-01 05:20:00,2.0,NaN


In [29]:
# [셀 5] 5분 완전 격자 + 0 채움  ★핵심 단계
# 원본엔 승객 0명인 5분 구간이 행으로 존재하지 않음
# → 빈틈없는 5분 시간표를 만들고, 기록 없는 칸은 0으로 채움
# 이게 없으면 다음 셀의 rolling(미래 합)이 시간을 건너뛰며 타깃을 오염시킴
full_idx = pd.date_range(raw.bording_date.min(),
                         raw.bording_date.max() + pd.Timedelta('23:55:00'),
                         freq='5min')
grid = wide.reindex(full_idx).fillna(0).astype(int)
grid.index.name = 'ts'

# 검증 2종
print('격자:', grid.shape, '(기대: 732일×288칸 =', 732 * 288, ')')
print('총합 보존:', grid.board.sum() + grid.alight.sum() == raw.qty.sum())

격자: (210816, 2) (기대: 732일×288칸 = 210816 )
총합 보존: True


In [30]:
# [셀 6] 타깃 8개 생성 + 운행창 필터
#
# (1) 타깃: 각 행 t에 "t 이후 5/15/30/60분간 승·하차 합" = 모델이 맞출 정답
#     rolling(n).sum() = 자기 포함 과거 n칸 합(방향 반대) → shift(-(n-1))로 미래 합으로 교정
data = grid.copy()
for col in ['board', 'alight']:
    for minutes, n in [(5, 1), (15, 3), (30, 6), (60, 12)]:
        data[f'y_{col}_{minutes}'] = grid[col].rolling(n).sum().shift(-(n - 1))

# 검증: 2024-03-04 18:00의 이후 60분 승차 = 897명이어야 함 (EDA에서 확인한 값)
assert data.loc['2024-03-04 18:00:00', 'y_board_60'] == 897, '타깃 계산 오류!'

# (2) 운행창 필터: 05:00 ~ 익일 00:10만 유지
#     근거: 공식 운행시간 05:30~24:20(대전교통공사) + 태그는 개찰 시점 기록이라
#           첫차 전 05:00부터 발생, 막차(24:20 도착) 하차 태그가 00:10까지 이어짐
#     제외: 04시대(활동일 1.6%, 노이즈), 00:15~01:35(연장운행 특이일), 02~04시(미운행)
#     ★ 반드시 타깃 생성 후에 잘라야 함 — 먼저 자르면 rolling이 빈 시간을 건너뛰어 타깃 오염
hhmm = data.index.strftime('%H:%M')
data = data[(hhmm >= '05:00') | (hhmm <= '00:10')]

# 타깃 계산 불가능한 꼬리(마지막 날 23:05 이후) 제거
data = data.dropna()

print(f'운행창 필터 후: {len(data):,}행 (하루 231칸 × 732일 근처)')
print(f'승차=0 행 비율: {(data.board == 0).mean():.1%}')

운행창 필터 후: 169,081행 (하루 231칸 × 732일 근처)
승차=0 행 비율: 2.9%


In [31]:
data 

bording_type,board,alight,y_board_5,y_board_15,y_board_30,y_board_60,y_alight_5,y_alight_15,y_alight_30,y_alight_60
ts,,,,,,,,,,
2024-01-01 00:00:00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 00:05:00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 00:10:00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-01-01 05:00:00,0,0,0.0,1.0,4.0,16.0,0.0,0.0,0.0,54.0
2024-01-01 05:05:00,0,0,0.0,1.0,9.0,26.0,0.0,0.0,0.0,65.0
...,...,...,...,...,...,...,...,...,...,...
2026-01-01 22:40:00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 22:45:00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026-01-01 22:50:00,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
# [셀 7] 기본 달력 피처 6개
# 타임스탬프를 모델이 먹을 수 있는 숫자 컬럼으로 분해
idx = data.index

data['hour']  = idx.hour                          # 시각 (0~23)
data['slot']  = idx.hour * 12 + idx.minute // 5   # 하루 내 5분 칸 번호 (0~287)
data['dow']   = idx.dayofweek                     # 요일 (0=월 ~ 6=일)
data['month'] = idx.month                         # 월 (계절성)
data['is_weekend'] = (data.dow >= 5).astype(int)

kr_hol = holidays.KR(years=range(2024, 2028), language='ko')     # 2027까지: 미래 추론 대비
hol = pd.to_datetime(list(kr_hol.keys()))
d = pd.Series(idx.normalize(), index=idx)         # 각 행의 '날짜'만 (아래 셀에서도 재사용)
data['is_holiday'] = d.isin(hol).astype(int)

print('공휴일 일수:', (data.groupby(d.dt.year).is_holiday.sum() / 231).round(1).to_dict())

공휴일 일수: {2024: 19.0, 2025: 19.0, 2026: 1.0}


In [34]:
# [셀 8] 확장 달력 피처 8개 — 피처 엔지니어링 ★
# 목표: "모든 휴일은 같다"는 뭉뚱그림을 깨서 특이일 예측 실패(성탄절 -24% 등)를 줄임
# 제약 준수: 전부 날짜 하나만으로 계산 가능 → 임의 미래 시점 추론에 사용 가능

# (1) 공휴일 세분화: 명절(설·추석 연휴)은 수요가 반토막, 일반 공휴일은 나들이 수요
#     → 전혀 다른 패턴인데 지금은 is_holiday 하나로 뭉쳐져 있음
major = pd.to_datetime([dt for dt, name in kr_hol.items()
                        if ('설날' in name) or ('추석' in name)])
data['is_major_holiday'] = d.isin(major).astype(int)                    # 명절 연휴
data['is_minor_holiday'] = (d.isin(hol) & ~d.isin(major)).astype(int)   # 그 외 공휴일

# (2) 연휴 구조: 같은 공휴일이라도 '연휴 첫날/끝날', '공휴일 전날 저녁' 패턴이 다름
data['holiday_prev'] = (d - pd.Timedelta('1D')).isin(hol).astype(int)   # 어제가 공휴일
data['holiday_next'] = (d + pd.Timedelta('1D')).isin(hol).astype(int)   # 내일이 공휴일(전야)

# (3) 샌드위치데이: 앞뒤가 다 쉬는 날인 평일 → 연차 내고 쉬는 사람 많아 수요 감소
off_prev = (d - pd.Timedelta('1D')).isin(hol) | ((d - pd.Timedelta('1D')).dt.dayofweek >= 5)
off_next = (d + pd.Timedelta('1D')).isin(hol) | ((d + pd.Timedelta('1D')).dt.dayofweek >= 5)
off_today = d.isin(hol) | (d.dt.dayofweek >= 5)
data['is_sandwich'] = (~off_today & off_prev & off_next).astype(int)

# (4) 연말 효과: 12/24~31은 달력상 평일이어도 수요 패턴이 다름 (성탄절 실패의 직접 처방)
data['is_yearend'] = ((d.dt.month == 12) & (d.dt.day >= 24)).astype(int)

# (5) 장기 추세: 2024→2025 완만한 증가(+2%) 반영. 기준일로부터 경과일수
#     주의: 트리는 학습 범위 밖 값을 외삽 못하므로, 먼 미래에선 "최근 수준"으로 포화됨(의도된 동작)
data['trend'] = (d - pd.Timestamp('2024-01-01')).dt.days

# --- 검증 ---
chk = data.groupby(d.dt.date)[['is_major_holiday','is_minor_holiday','is_sandwich','is_yearend']].max()
print('명절 연휴 일수/년:', chk.is_major_holiday.groupby(pd.to_datetime(chk.index).year).sum().to_dict())
print('샌드위치데이:', list(chk[chk.is_sandwich == 1].index)[:8])
print('피처 수:', len([c for c in data.columns if not c.startswith('y_') and c not in ['board','alight']]))

명절 연휴 일수/년: {2024: 7, 2025: 7, 2026: 0}
샌드위치데이: [datetime.date(2024, 6, 7), datetime.date(2024, 8, 16), datetime.date(2024, 9, 30), datetime.date(2024, 10, 2), datetime.date(2024, 10, 4), datetime.date(2025, 1, 31), datetime.date(2025, 6, 2), datetime.date(2025, 10, 10)]
피처 수: 13


In [39]:
# [셀 9] train / val / test 분할 저장
# ★ 시간순 분할 (셔플 금지): 랜덤 분할은 미래로 과거를 맞추는 컨닝이 되어 성능이 부풀려짐
#   train 2024-01-01 ~ 2025-06-30 (18개월) : 학습
#   val   2025-07-01 ~ 2025-09-30 (3개월)  : 조기종료·실험 비교용
#   test  2025-10-01 ~ 2025-12-31 (3개월)  : 최종 채점용 (마지막에 한 번만)
#   2026-01-01은 덤프 경계일(기록 없는 가짜 0)이라 제외

train = data[:'2025-06-30 23:55:00']
val   = data['2025-07-01':'2025-09-30 23:55:00']
test  = data['2025-10-01':'2025-12-31 23:55:00']

train.to_csv(f'{OUT_DIR}\\train.csv')
val.to_csv(f'{OUT_DIR}\\val.csv')
test.to_csv(f'{OUT_DIR}\\test.csv')

for name, part in [('train', train), ('val', val), ('test', test)]:
    print(f'{name:5s} {len(part):>7,}행  {part.index.min()} ~ {part.index.max()}')
print('제외된 행(2026-01-01):', len(data) - len(train) - len(val) - len(test))

train 126,357행  2024-01-01 00:00:00 ~ 2025-06-30 23:55:00
val    21,252행  2025-07-01 00:00:00 ~ 2025-09-30 23:55:00
test   21,252행  2025-10-01 00:00:00 ~ 2025-12-31 23:55:00
제외된 행(2026-01-01): 220
